# Serve Centaur and the tutor from Colab (for `centaur_main`)

**Why.** With both models loaded, the laptop runs out of RAM (~9 GB of shared GPU memory), Windows pages the
weights to disk and Centaur runs ~7x slower (measured 2026-09-18: up to 61,000 hard page faults per second). This
notebook serves the **same GGUF files** (SHA-256 checked against the laptop copies) with llama.cpp's
`llama-server` on a Colab GPU, behind a small proxy that reproduces **LM Studio's exact prompt rendering**, and
exposes it through a Cloudflare quick tunnel. The laptop keeps running `simulate.py` and only makes HTTP calls.

**What the proxy reproduces** (captured from LM Studio with `lms log stream`, 2026-09-18):

- Centaur's GGUF has no chat template, so LM Studio feeds `<|begin_of_text|>AI: ` + the transcript. The proxy sends
  `AI: ` + transcript to `/completion`, where llama-server inserts the BOS token itself, and returns the raw-softmax
  top logprobs in the OpenAI chat format `engines.MinitaurEngine` reads.
- Qwen gets the ChatML its GGUF template produces (identical to LM Studio's rendering); sampling parameters the
  tutor does not set take LM Studio's defaults.

**Steps**

1. Runtime → Change runtime type → **L4** (a T4 works too, the build is just slower).
2. Secrets (key icon, notebook access on): `HF_TOKEN` with read access to `MatteoG4444/Llama-3.1-Centaur-8B-GGUF`.
3. Runtime → Run all. The first run downloads ~7 GB and builds llama.cpp (~5-15 min).
4. The tunnel cell prints two lines: paste them into the laptop's `.env`, replacing older ones.
5. On the laptop: `uv run python scripts/compare_servers.py` must report **PASS** before the run switches.
6. Keep this tab open while the run lasts. The last cell prints request counts every minute.

If the session drops, run all again (downloads and the build are redone) and paste the new two lines into `.env`;
the laptop loop retries until the server is back.

In [ ]:
# Configuration. The SHA-256 values are the laptop's LM Studio copies (Get-FileHash, 2026-09-18).
CENTAUR = {"repo": "MatteoG4444/Llama-3.1-Centaur-8B-GGUF", "file": "llama-3.1-centaur-8b-q4_k_m.gguf",
           "sha256": "d94595b8ff66789fa6108cee731134b8ff27ccc5abb2d0c522547ed9e0e9bb8b",
           "alias": "llama-3.1-centaur-8b", "port": 8081, "ctx": 8192}
QWEN = {"repo": "Qwen/Qwen2.5-3B-Instruct-GGUF", "file": "qwen2.5-3b-instruct-q4_k_m.gguf",
        "sha256": "626b4a6678b86442240e33df819e00132d3ba7dddfe1cdc4fbb18e0a9615c62d",
        "alias": "qwen2.5-3b-instruct", "port": 8082, "ctx": 8192}
LLAMA_CPP_TAG = "v0.4.1"                                          # latest release on 2026-09-18
LLAMA_CPP_COMMIT = "391fac16460f15233a7740550d858ac96df3419d"
CENTAUR_PREFIX = "AI: "   # LM Studio's rendering of the assistant prefill for a GGUF without a chat template
DOUBLE_BOS = False        # True only if scripts/compare_servers.py finds LM Studio feeds two BOS tokens
TUTOR_DEFAULTS = {"top_k": 40, "top_p": 0.95, "min_p": 0.05, "repeat_penalty": 1.1}  # LM Studio defaults
PROXY_PORT = 8000
MODELS_DIR = "/content/models"

In [ ]:
# Environment: an NVIDIA GPU and the Hugging Face token
import hashlib, json, math, os, re, secrets, subprocess, threading, time
import requests

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "No GPU: Runtime -> Change runtime type -> L4 (or T4)"
GPU_NAME, GPU_MEMORY, COMPUTE_CAP = [x.strip() for x in gpu.stdout.strip().splitlines()[0].split(",")]
CUDA_ARCH = COMPUTE_CAP.replace(".", "")
print(f"{GPU_NAME}, {GPU_MEMORY}, compute capability {COMPUTE_CAP}, {os.cpu_count()} CPUs")

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "add the HF_TOKEN secret (read access to the Centaur GGUF repository)"


def sh(cmd, log=None):
    """Run a shell command; on failure print the tail of its log and stop."""
    print("$", cmd)
    result = subprocess.run(cmd if log is None else f"{cmd} > {log} 2>&1", shell=True)
    if result.returncode != 0:
        if log:
            print(open(log).read()[-4000:])
        raise RuntimeError(f"command failed ({result.returncode}): {cmd}")

In [ ]:
# Download both GGUF files and verify they are byte-identical to the laptop copies
from huggingface_hub import hf_hub_download


def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


PATHS = {}
for m in (CENTAUR, QWEN):
    started = time.time()
    path = hf_hub_download(m["repo"], m["file"], token=HF_TOKEN, local_dir=MODELS_DIR)
    digest = sha256(path)
    assert digest == m["sha256"], f"{m['file']}: sha256 {digest} differs from the laptop copy {m['sha256']}"
    PATHS[m["alias"]] = path
    print(f"ok {m['file']}: {os.path.getsize(path) / 1e9:.2f} GB, sha256 matches the laptop ({time.time() - started:.0f}s)")

In [ ]:
# Build llama-server at the pinned commit with CUDA for this GPU only (the build is the slow part)
LLAMA_DIR = "/content/llama.cpp"
SERVER_BIN = f"{LLAMA_DIR}/build/bin/llama-server"
if not os.path.exists(SERVER_BIN):
    if not os.path.exists(LLAMA_DIR):
        sh(f"git clone --filter=blob:none --quiet https://github.com/ggml-org/llama.cpp {LLAMA_DIR}")
    sh(f"git -C {LLAMA_DIR} checkout --quiet {LLAMA_CPP_COMMIT}")
    started = time.time()
    sh(f"cmake -S {LLAMA_DIR} -B {LLAMA_DIR}/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH} "
       f"-DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF",
       log="/content/cmake_configure.log")
    sh(f"cmake --build {LLAMA_DIR}/build --config Release -j {os.cpu_count()} --target llama-server",
       log="/content/cmake_build.log")
    print(f"built in {time.time() - started:.0f}s")
COMMIT = subprocess.run(["git", "-C", LLAMA_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert COMMIT == LLAMA_CPP_COMMIT, f"llama.cpp is at {COMMIT}, expected {LLAMA_CPP_COMMIT}"
version = subprocess.run([SERVER_BIN, "--version"], capture_output=True, text=True)
print((version.stdout + version.stderr).strip()[-400:])

In [ ]:
# Start one llama-server per model: all layers on the GPU, one slot (the laptop calls sequentially)
PROCS = {}


def start_server(m):
    if m["alias"] in PROCS and PROCS[m["alias"]].poll() is None:
        PROCS[m["alias"]].terminate()
        PROCS[m["alias"]].wait()
    log = open(f"/content/{m['alias']}.log", "w")
    cmd = [SERVER_BIN, "-m", PATHS[m["alias"]], "--alias", m["alias"], "-c", str(m["ctx"]), "-ngl", "999",
           "-np", "1", "--host", "127.0.0.1", "--port", str(m["port"])]
    PROCS[m["alias"]] = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(300):
        if PROCS[m["alias"]].poll() is not None:
            raise RuntimeError(open(log.name).read()[-4000:])
        try:
            if requests.get(f"http://127.0.0.1:{m['port']}/health", timeout=2).json().get("status") == "ok":
                print(f"{m['alias']} ready on port {m['port']}")
                return
        except (requests.ConnectionError, requests.Timeout, ValueError):
            pass
        time.sleep(2)
    raise RuntimeError(f"{m['alias']} did not become healthy; see {log.name}")


for m in (CENTAUR, QWEN):
    start_server(m)

In [ ]:
# The proxy: OpenAI-compatible routes for the laptop, with LM Studio's rendering for Centaur
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

TOKEN = globals().get("TOKEN") or secrets.token_urlsafe(24)  # kept across re-runs so .env stays valid
STATS = {"centaur": 0, "tutor": 0, "raw": 0, "errors": 0, "last_error": None, "started": time.time()}
LOCK = threading.Lock()
META = {"llama_cpp_tag": LLAMA_CPP_TAG, "llama_cpp_commit": COMMIT, "gpu": GPU_NAME, "compute_capability": COMPUTE_CAP,
        "models": {m["alias"]: {"file": m["file"], "sha256": m["sha256"], "ctx": m["ctx"]} for m in (CENTAUR, QWEN)},
        "centaur_prefix": CENTAUR_PREFIX, "double_bos": DOUBLE_BOS, "tutor_defaults": TUTOR_DEFAULTS,
        "server_flags": "-ngl 999 -np 1", "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}


def count(key, error=None):
    with LOCK:
        STATS[key] += 1
        if error:
            STATS["errors"] += 1
            STATS["last_error"] = f"{time.strftime('%H:%M:%S')} {error[:300]}"


def centaur_chat(body):
    msgs = body.get("messages") or []
    if len(msgs) != 1 or msgs[0].get("role") != "assistant":
        return 400, {"error": "the Centaur route takes exactly one assistant message: the transcript to continue"}
    prompt = ("<|begin_of_text|>" if DOUBLE_BOS else "") + CENTAUR_PREFIX + msgs[0]["content"]
    req = {"prompt": prompt, "n_predict": int(body.get("max_tokens", 1)),
           "n_probs": int(body.get("top_logprobs", 20)) if body.get("logprobs") else 0,
           "temperature": float(body.get("temperature", 0.0)), "cache_prompt": True, "post_sampling_probs": False}
    r = requests.post(f"http://127.0.0.1:{CENTAUR['port']}/completion", json=req, timeout=600)
    if r.status_code != 200:
        return r.status_code, {"error": r.text[:1000]}
    d = r.json()
    if d.get("truncated"):
        return 400, {"error": f"prompt of {d.get('tokens_evaluated')} tokens exceeds the {CENTAUR['ctx']}-token context"}
    content = []
    for p in d.get("completion_probabilities") or d.get("probs") or []:
        tops = p.get("top_logprobs")
        if tops is None:  # post-sampling format; not requested, handled for safety
            tops = [{"token": t["token"], "logprob": math.log(max(t["prob"], 1e-45))} for t in p.get("top_probs", [])]
        content.append({"token": p.get("token"), "logprob": p.get("logprob"),
                        "top_logprobs": [{"token": t["token"], "logprob": t["logprob"]} for t in tops]})
    n_prompt = d.get("tokens_evaluated")
    return 200, {"id": f"chatcmpl-{secrets.token_hex(6)}", "object": "chat.completion", "created": int(time.time()),
                 "model": CENTAUR["alias"],
                 "choices": [{"index": 0, "message": {"role": "assistant", "content": d.get("content", "")},
                              "logprobs": {"content": content}, "finish_reason": "length"}],
                 "usage": {"prompt_tokens": n_prompt, "completion_tokens": len(content),
                           "total_tokens": (n_prompt or 0) + len(content)},
                 "render": {"prefix": CENTAUR_PREFIX, "double_bos": DOUBLE_BOS, "tokens_cached": d.get("tokens_cached")}}


def tutor_chat(body):
    r = requests.post(f"http://127.0.0.1:{QWEN['port']}/v1/chat/completions", json={**TUTOR_DEFAULTS, **body}, timeout=600)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, {"error": r.text[:1000]}


RAW = {"/raw/centaur/completion": (CENTAUR, "/completion"), "/raw/centaur/tokenize": (CENTAUR, "/tokenize"),
       "/raw/qwen/apply-template": (QWEN, "/apply-template")}


class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *args):
        pass

    def send_json(self, code, obj):
        body = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def authorised(self):
        if self.headers.get("Authorization", "") == f"Bearer {TOKEN}":
            return True
        self.send_json(401, {"error": "missing or wrong bearer token (NEUROTUTOR_SERVER_TOKEN)"})
        return False

    def do_GET(self):
        if not self.authorised():
            return
        path = self.path.rstrip("/")
        if path in ("/v1/models", "/models"):
            return self.send_json(200, {"object": "list", "data": [{"id": m["alias"], "object": "model"} for m in (CENTAUR, QWEN)]})
        if path == "/meta":
            return self.send_json(200, META)
        if path == "/health":
            return self.send_json(200, {"status": "ok", **STATS})
        self.send_json(404, {"error": f"no route {self.path}"})

    def do_POST(self):
        if not self.authorised():
            return
        try:
            body = json.loads(self.rfile.read(int(self.headers.get("Content-Length") or 0)) or b"{}")
            if self.path.rstrip("/") == "/v1/chat/completions":
                model = body.get("model")
                if model == CENTAUR["alias"]:
                    code_, obj = centaur_chat(body)
                    count("centaur", None if code_ == 200 else str(obj))
                elif model == QWEN["alias"]:
                    code_, obj = tutor_chat(body)
                    count("tutor", None if code_ == 200 else str(obj))
                else:
                    code_, obj = 404, {"error": f"unknown model {model!r}"}
                return self.send_json(code_, obj)
            if self.path in RAW:
                m, route = RAW[self.path]
                r = requests.post(f"http://127.0.0.1:{m['port']}{route}", json=body, timeout=600)
                count("raw")
                return self.send_json(r.status_code, r.json())
            self.send_json(404, {"error": f"no route {self.path}"})
        except Exception as exc:  # noqa: BLE001 - report every failure to the caller and in the stats
            count("raw", repr(exc))
            self.send_json(500, {"error": repr(exc)})


if "PROXY" in globals():
    PROXY.shutdown()
    PROXY.server_close()
PROXY = ThreadingHTTPServer(("127.0.0.1", PROXY_PORT), Handler)
threading.Thread(target=PROXY.serve_forever, daemon=True).start()
hdr = {"Authorization": f"Bearer {TOKEN}"}
print(requests.get(f"http://127.0.0.1:{PROXY_PORT}/v1/models", headers=hdr, timeout=10).json())

In [ ]:
# Cloudflare quick tunnel (no account needed), then a self-test through it
if not os.path.exists("/content/cloudflared"):
    sh("wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
       " && chmod +x /content/cloudflared")
if "TUNNEL" in globals() and TUNNEL.poll() is None:
    TUNNEL.terminate()
    TUNNEL.wait()
open("/content/cloudflared.log", "w").close()
TUNNEL = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PROXY_PORT}", "--no-autoupdate"],
                          stdout=open("/content/cloudflared.log", "a"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if found:
        URL = found.group(0)
        break
    time.sleep(2)
assert URL, open("/content/cloudflared.log").read()[-3000:]

probe = "Problem 1.\nQuestion: What is 2 + 2?\nOptions: Q) 4  W) 5\nYou press <<"
for attempt in range(20):  # a new quick-tunnel hostname can take a few seconds to resolve
    try:
        started = time.time()
        r = requests.post(f"{URL}/v1/chat/completions", headers=hdr, timeout=120,
                          json={"model": CENTAUR["alias"], "messages": [{"role": "assistant", "content": probe}],
                                "max_tokens": 1, "temperature": 0.0, "logprobs": True, "top_logprobs": 5})
        r.raise_for_status()
        tops = r.json()["choices"][0]["logprobs"]["content"][0]["top_logprobs"]
        print("centaur via tunnel:", [(t["token"], round(math.exp(t["logprob"]), 3)) for t in tops],
              f"{time.time() - started:.1f}s")
        break
    except Exception as exc:  # noqa: BLE001
        print("waiting for the tunnel:", repr(exc)[:120])
        time.sleep(5)
else:
    raise RuntimeError("the tunnel never answered; rerun this cell")
started = time.time()
r = requests.post(f"{URL}/v1/chat/completions", headers=hdr, timeout=120,
                  json={"model": QWEN["alias"], "temperature": 0.3, "seed": 7, "max_tokens": 40,
                        "messages": [{"role": "system", "content": "You are a tutor."},
                                     {"role": "user", "content": "Say hello in five words."}]})
print("tutor via tunnel:", r.json()["choices"][0]["message"]["content"][:120], f"{time.time() - started:.1f}s")
print("\nPaste these two lines into the laptop's .env (replace older ones), then run scripts/compare_servers.py:\n")
print(f"NEUROTUTOR_SERVER_URL={URL}")
print(f"NEUROTUTOR_SERVER_TOKEN={TOKEN}")

In [ ]:
# Keep-alive monitor: request counts every minute. Stop it (the square button) before re-running earlier cells.
try:
    while True:
        time.sleep(60)
        alive = {name: p.poll() is None for name, p in PROCS.items()}
        alive["tunnel"] = TUNNEL.poll() is None
        up = (time.time() - STATS["started"]) / 60
        print(f"{time.strftime('%H:%M:%S')} up {up:.0f} min | centaur {STATS['centaur']} tutor {STATS['tutor']} "
              f"raw {STATS['raw']} errors {STATS['errors']} | alive {alive}"
              + (f" | last error {STATS['last_error']}" if STATS["last_error"] else ""))
        if not all(alive.values()):
            print("SOMETHING STOPPED: rerun the cells from 'Start one llama-server per model' and paste the new lines into .env")
except KeyboardInterrupt:
    print("monitor stopped; the servers and the tunnel keep running")